# Environment Tool
输入火点经纬度，一次返回地形、天气、土地覆盖、水源和道路信息。

In [10]:
# 1. 参数
from pathlib import Path

LATITUDE = 32.0725
LONGITUDE = 118.8406
DEM_PATH = Path("C:/Users/19565/000jupyter/无人机火情/N32E118.hgt")# 修改为本地 SRTM DEM 路径
WATER_RADIUS_M = 3000
ROAD_RADIUS_M = 3000

In [11]:
# 2. 依赖与公共函数
import json
import math
import time
from collections import Counter

import numpy as np
import requests
import rasterio
import planetary_computer as pc

from pyproj import Transformer
from shapely.geometry import Point, LineString
from pystac_client import Client
from rasterio.windows import Window
from rasterio.transform import xy
from rasterio.warp import transform


WORLD_COVER_CLASSES = {
    10: "Tree Cover", 20: "Shrubland", 30: "Grassland",
    40: "Cropland", 50: "Built-up", 60: "Bare / Sparse Vegetation",
    70: "Snow and Ice", 80: "Permanent Water Bodies",
    90: "Herbaceous Wetland", 95: "Mangroves", 100: "Moss and Lichen",
}
BURNABLE_NATURAL = {10, 20, 30}
PREFERRED_WATER_TYPES = {"reservoir", "lake", "pond"}

ALL_HIGHWAY_TYPES = {
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "unclassified", "residential", "service", "track",
    "path", "footway", "steps", "cycleway",
}
VEHICLE_HIGHWAY_TYPES = {
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "unclassified", "residential", "service", "track",
}

OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"
OVERPASS_URL = "https://overpass-api.de/api/interpreter"


def validate_coordinates(latitude, longitude):
    try:
        latitude = float(latitude)
        longitude = float(longitude)
    except (TypeError, ValueError):
        raise ValueError("经纬度必须是数字")

    if not -90 <= latitude <= 90:
        raise ValueError("latitude 必须位于 -90 到 90 之间")
    if not -180 <= longitude <= 180:
        raise ValueError("longitude 必须位于 -180 到 180 之间")
    return latitude, longitude


def degree_to_direction(deg):
    if deg is None:
        return None
    directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    return directions[int((float(deg) + 22.5) // 45) % 8]


def get_utm_epsg(latitude, longitude):
    zone = int((longitude + 180) // 6) + 1
    return (32600 if latitude >= 0 else 32700) + zone


def haversine_distance_m(lat1, lon1, lat2, lon2):
    r = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return r * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def overpass_query(query, timeout=30, retries=2):
    last_error = None
    for attempt in range(retries):
        try:
            r = requests.post(
                OVERPASS_URL,
                data={"data": query},
                headers={"User-Agent": "ForestFire-EnvironmentTool/1.0"},
                timeout=timeout,
            )
            r.raise_for_status()
            return r.json().get("elements", [])
        except requests.RequestException as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(2 ** (attempt + 1))
    raise RuntimeError(f"Overpass 请求失败: {last_error}")


def safe_call(func, *args, **kwargs):
    try:
        return {"status": "ok", **func(*args, **kwargs)}
    except Exception as exc:
        return {"status": "error", "error": str(exc)}


In [12]:
# 3. 地形
def get_terrain(latitude, longitude, dem_path, window_size=5):
    dem_path = Path(dem_path)
    if not dem_path.is_file():
        raise FileNotFoundError(f"未找到 DEM 文件: {dem_path}")

    with rasterio.open(dem_path) as ds:
        if ds.crs is None:
            raise RuntimeError("DEM 缺少 CRS")

        to_dem = Transformer.from_crs("EPSG:4326", ds.crs, always_xy=True)
        x0, y0 = to_dem.transform(longitude, latitude)

        if not (ds.bounds.left <= x0 <= ds.bounds.right and ds.bounds.bottom <= y0 <= ds.bounds.top):
            raise ValueError("输入坐标不在当前 DEM 覆盖范围内")

        row, col = ds.index(x0, y0)
        half = window_size // 2
        window = Window(col - half, row - half, window_size, window_size)

        data = ds.read(1, window=window, boundless=True, masked=True).astype(float)
        center = data[data.shape[0] // 2, data.shape[1] // 2]
        if np.ma.is_masked(center):
            raise RuntimeError("中心像素没有有效高程值")

        rows, cols = np.indices(data.shape)
        xs, ys = xy(ds.window_transform(window), rows, cols, offset="center")

        z = data.filled(np.nan).ravel()
        x = np.asarray(xs, dtype=float).ravel()
        y = np.asarray(ys, dtype=float).ravel()
        valid = np.isfinite(z) & (~np.ma.getmaskarray(data).ravel())

        if valid.sum() < 3:
            raise RuntimeError("有效 DEM 像素不足")

        to_wgs84 = Transformer.from_crs(ds.crs, "EPSG:4326", always_xy=True)
        lons, lats = to_wgs84.transform(x[valid], y[valid])

        epsg = get_utm_epsg(latitude, longitude)
        to_utm = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
        east, north = to_utm.transform(lons, lats)

        east = np.asarray(east, dtype=float)
        north = np.asarray(north, dtype=float)
        east -= east.mean()
        north -= north.mean()

        A = np.column_stack([east, north, np.ones_like(east)])
        dz_dx, dz_dy, _ = np.linalg.lstsq(A, z[valid], rcond=None)[0]

        slope_deg = math.degrees(math.atan(math.hypot(dz_dx, dz_dy)))

        if slope_deg < 0.1:
            down_deg = up_deg = None
        else:
            down_deg = (math.degrees(math.atan2(-dz_dx, -dz_dy)) + 360) % 360
            up_deg = (down_deg + 180) % 360

    return {
        "elevation_m": round(float(center), 1),
        "slope_deg": round(slope_deg, 2),
        "downslope_deg": None if down_deg is None else round(down_deg, 1),
        "downslope_direction": degree_to_direction(down_deg),
        "upslope_deg": None if up_deg is None else round(up_deg, 1),
        "upslope_direction": degree_to_direction(up_deg),
    }


In [13]:
# 4. 天气
def get_weather(latitude, longitude):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "wind_speed_10m",
            "wind_direction_10m",
            "wind_gusts_10m",
        ]),
        "timezone": "auto",
        "wind_speed_unit": "ms",
    }

    r = requests.get(OPEN_METEO_URL, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()
    current = data.get("current", {})

    wind_from_deg = current.get("wind_direction_10m")
    wind_to_deg = None if wind_from_deg is None else (float(wind_from_deg) + 180) % 360

    return {
        "time": current.get("time"),
        "temperature_c": current.get("temperature_2m"),
        "relative_humidity_pct": current.get("relative_humidity_2m"),
        "precipitation_mm": current.get("precipitation"),
        "wind_speed_m_s": current.get("wind_speed_10m"),
        "wind_from_deg": wind_from_deg,
        "wind_from_direction": degree_to_direction(wind_from_deg),
        "wind_to_deg": None if wind_to_deg is None else round(wind_to_deg, 1),
        "wind_to_direction": degree_to_direction(wind_to_deg),
        "wind_gust_m_s": current.get("wind_gusts_10m"),
        "timezone": data.get("timezone"),
    }


In [14]:
# 5. 土地覆盖
def get_landcover(latitude, longitude, window_size=11, min_burnable_ratio=0.4):
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    search = catalog.search(
        collections=["esa-worldcover"],
        intersects={"type": "Point", "coordinates": [longitude, latitude]},
    )
    item = next(search.items(), None)
    if item is None:
        raise RuntimeError("当前位置没有找到 ESA WorldCover 数据")

    asset = item.assets.get("map") or next(iter(item.assets.values()))

    with rasterio.open(asset.href) as ds:
        xs, ys = transform("EPSG:4326", ds.crs, [longitude], [latitude])
        row, col = ds.index(xs[0], ys[0])

        center = int(ds.read(
            1,
            window=Window(col, row, 1, 1),
            boundless=True,
            fill_value=0,
        )[0, 0])

        half = window_size // 2
        data = ds.read(
            1,
            window=Window(col - half, row - half, window_size, window_size),
            boundless=True,
            fill_value=0,
        )

    valid = data[data != 0]
    if valid.size == 0:
        raise RuntimeError("WorldCover 返回窗口中没有有效像素")

    counts = Counter(valid.ravel().tolist())
    dominant = int(counts.most_common(1)[0][0])
    burnable_ratio = float(np.isin(valid, list(BURNABLE_NATURAL)).sum() / valid.size)

    return {
        "center_class": WORLD_COVER_CLASSES.get(center, "Unknown"),
        "dominant_class": WORLD_COVER_CLASSES.get(dominant, "Unknown"),
        "burnable_ratio": round(burnable_ratio, 3),
        "fuel_possible": burnable_ratio >= min_burnable_ratio,
    }


In [15]:
# 6. 水源
def classify_water(tags):
    if tags.get("landuse") == "reservoir" or tags.get("water") == "reservoir":
        return "reservoir"
    if tags.get("water") in {"lake", "pond"}:
        return tags["water"]
    if tags.get("waterway") in {"river", "stream"}:
        return tags["waterway"]
    return tags.get("water") or "water"


def water_name(tags, source_type):
    name = tags.get("name:zh") or tags.get("name") or tags.get("ref")
    if name:
        return name
    return {
        "stream": "未命名溪流", "river": "未命名河流",
        "reservoir": "未命名水库", "lake": "未命名湖泊",
        "pond": "未命名池塘", "water": "未命名水体",
    }.get(source_type, "未命名水体")


def get_water_sources(latitude, longitude, search_radius_m=3000):
    query = f'''
    [out:json][timeout:25];
    (
      nwr(around:{search_radius_m},{latitude},{longitude})["natural"="water"];
      nwr(around:{search_radius_m},{latitude},{longitude})["landuse"="reservoir"];
      nwr(around:{search_radius_m},{latitude},{longitude})["waterway"="river"];
      nwr(around:{search_radius_m},{latitude},{longitude})["waterway"="stream"];
    );
    out tags center;
    '''

    features = []
    for e in overpass_query(query):
        tags = e.get("tags", {})
        center = e if "lat" in e and "lon" in e else e.get("center", {})
        lat, lon = center.get("lat"), center.get("lon")
        if lat is None or lon is None:
            continue

        source_type = classify_water(tags)
        features.append({
            "name": water_name(tags, source_type),
            "type": source_type,
            "latitude": round(float(lat), 6),
            "longitude": round(float(lon), 6),
            "distance_m": round(haversine_distance_m(latitude, longitude, lat, lon), 1),
        })

    features.sort(key=lambda x: x["distance_m"])
    preferred = [x for x in features if x["type"] in PREFERRED_WATER_TYPES]

    return {
        "found": bool(features),
        "feature_count": len(features),
        "nearest": features[0] if features else None,
        "preferred": preferred[0] if preferred else None,
        "distance_note": "到 OSM 水体代表点的近似距离",
    }


In [16]:
# 7. 道路
def road_name(tags):
    name = tags.get("name:zh") or tags.get("name") or tags.get("ref")
    if name:
        return name
    return {
        "service": "未命名服务道路",
        "track": "未命名林区/土路",
        "path": "未命名小径",
        "footway": "未命名步道",
        "steps": "未命名台阶",
    }.get(tags.get("highway"), "未命名道路")


def line_distance(latitude, longitude, geometry):
    if not geometry or len(geometry) < 2:
        return None

    epsg = get_utm_epsg(latitude, longitude)
    to_utm = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    to_wgs84 = Transformer.from_crs(f"EPSG:{epsg}", "EPSG:4326", always_xy=True)

    fire_x, fire_y = to_utm.transform(longitude, latitude)
    road_xy = [to_utm.transform(p["lon"], p["lat"]) for p in geometry if "lat" in p and "lon" in p]
    if len(road_xy) < 2:
        return None

    line = LineString(road_xy)
    fire = Point(fire_x, fire_y)
    nearest = line.interpolate(line.project(fire))
    near_lon, near_lat = to_wgs84.transform(nearest.x, nearest.y)

    return {
        "distance_m": round(float(fire.distance(line)), 1),
        "nearest_point": {
            "latitude": round(float(near_lat), 6),
            "longitude": round(float(near_lon), 6),
        },
    }


def get_road_context(latitude, longitude, search_radius_m=3000):
    highway_regex = "|".join(sorted(ALL_HIGHWAY_TYPES))
    query = f'''
    [out:json][timeout:25];
    way(around:{search_radius_m},{latitude},{longitude})
      ["highway"~"^({highway_regex})$"];
    out tags geom;
    '''

    roads = []
    for e in overpass_query(query):
        tags = e.get("tags", {})
        distance = line_distance(latitude, longitude, e.get("geometry", []))
        if distance is None:
            continue

        roads.append({
            "name": road_name(tags),
            "highway": tags.get("highway"),
            "surface": tags.get("surface"),
            "access": tags.get("access"),
            "motor_vehicle": tags.get("motor_vehicle"),
            **distance,
        })

    roads.sort(key=lambda x: x["distance_m"])
    vehicle = [
        r for r in roads
        if r["highway"] in VEHICLE_HIGHWAY_TYPES
        and r["access"] != "no"
        and r["motor_vehicle"] != "no"
    ]

    return {
        "found": bool(roads),
        "road_count": len(roads),
        "nearest_transport": roads[0] if roads else None,
        "nearest_vehicle_access": vehicle[0] if vehicle else None,
    }


In [17]:
# 8. 总入口
def get_environment(
    latitude,
    longitude,
    dem_path=None,
    water_radius_m=3000,
    road_radius_m=3000,
):
    try:
        latitude, longitude = validate_coordinates(latitude, longitude)
    except ValueError as exc:
        return {
            "status": "invalid_input",
            "error": str(exc),
            "location": {"latitude": latitude, "longitude": longitude},
        }

    dem_path = Path(dem_path) if dem_path is not None else DEM_PATH

    modules = {
        "terrain": safe_call(get_terrain, latitude, longitude, dem_path),
        "weather": safe_call(get_weather, latitude, longitude),
        "landcover": safe_call(get_landcover, latitude, longitude),
        "water": safe_call(get_water_sources, latitude, longitude, water_radius_m),
        "road": safe_call(get_road_context, latitude, longitude, road_radius_m),
    }

    ok_count = sum(v["status"] == "ok" for v in modules.values())
    status = "ok" if ok_count == len(modules) else ("partial" if ok_count else "error")

    return {
        "status": status,
        "location": {
            "latitude": latitude,
            "longitude": longitude,
        },
        **modules,
        "sources": {
            "terrain": "NASA SRTM",
            "weather": "Open-Meteo",
            "landcover": "ESA WorldCover / Microsoft Planetary Computer",
            "water": "OpenStreetMap / Overpass API",
            "road": "OpenStreetMap / Overpass API",
        },
    }


def print_environment_brief(result):
    if result["status"] == "invalid_input":
        print("输入错误：", result["error"])
        return

    print("总体状态：", result["status"])

    t = result["terrain"]
    if t["status"] == "ok":
        print(f"地形：海拔 {t['elevation_m']} m，坡度 {t['slope_deg']}°，上坡 {t['upslope_direction']}")
    else:
        print("地形：失败 -", t["error"])

    w = result["weather"]
    if w["status"] == "ok":
        print(
            f"天气：{w['temperature_c']} ℃，湿度 {w['relative_humidity_pct']}%，"
            f"风速 {w['wind_speed_m_s']} m/s，{w['wind_from_direction']} → {w['wind_to_direction']}"
        )
    else:
        print("天气：失败 -", w["error"])

    lc = result["landcover"]
    if lc["status"] == "ok":
        print(
            f"植被：{lc['dominant_class']}，可燃覆盖比例 {lc['burnable_ratio']:.0%}"
        )
    else:
        print("植被：失败 -", lc["error"])

    water = result["water"]
    if water["status"] == "ok":
        source = water["preferred"] or water["nearest"]
        if source:
            print(f"水源：{source['name']}，约 {source['distance_m']} m")
        else:
            print("水源：搜索范围内未找到")
    else:
        print("水源：失败 -", water["error"])

    road = result["road"]
    if road["status"] == "ok":
        access = road["nearest_vehicle_access"] or road["nearest_transport"]
        if access:
            print(f"道路：{access['name']}，约 {access['distance_m']} m")
        else:
            print("道路：搜索范围内未找到")
    else:
        print("道路：失败 -", road["error"])


In [18]:
# 9. 运行
environment = get_environment(
    LATITUDE,
    LONGITUDE,
    dem_path=DEM_PATH,
    water_radius_m=WATER_RADIUS_M,
    road_radius_m=ROAD_RADIUS_M,
)

print_environment_brief(environment)

print("\n完整 JSON：")
print(json.dumps(environment, ensure_ascii=False, indent=2))


总体状态： ok
地形：海拔 427.0 m，坡度 13.07°，上坡 SE
天气：23.0 ℃，湿度 85%，风速 5.78 m/s，NE → SW
植被：Tree Cover，可燃覆盖比例 100%
水源：紫霞湖，约 1151.8 m
道路：未命名服务道路，约 33.8 m

完整 JSON：
{
  "status": "ok",
  "location": {
    "latitude": 32.0725,
    "longitude": 118.8406
  },
  "terrain": {
    "status": "ok",
    "elevation_m": 427.0,
    "slope_deg": 13.07,
    "downslope_deg": 313.8,
    "downslope_direction": "NW",
    "upslope_deg": 133.8,
    "upslope_direction": "SE"
  },
  "weather": {
    "status": "ok",
    "time": "2026-09-03T11:15",
    "temperature_c": 23.0,
    "relative_humidity_pct": 85,
    "precipitation_mm": 0.0,
    "wind_speed_m_s": 5.78,
    "wind_from_deg": 37,
    "wind_from_direction": "NE",
    "wind_to_deg": 217.0,
    "wind_to_direction": "SW",
    "wind_gust_m_s": 13.3,
    "timezone": "Asia/Shanghai"
  },
  "landcover": {
    "status": "ok",
    "center_class": "Tree Cover",
    "dominant_class": "Tree Cover",
    "burnable_ratio": 1.0,
    "fuel_possible": true
  },
  "water": {
    "statu